# Tarea 2 – Parte 2: Evaluación y extracción estructurada

## 2.1. Muestra de evaluación (sin nuevos datos)

*De los 25 documentos de la Parte 1, seleccionen 10 (incluyendo los tres tipos); y de sus 20 clips o más, seleccionen 10 (con distintas fuentes ocondiciones de grabación). Realicen una transcripción manual de esos mismos archivos y completensu columna texto_referencia en los CSV. No es necesario recolectar otros documentos ni audios. Usen exactamente esos 10 casos de cada modalidad para evaluar todos los modelos, y definan susconfiguraciones antes de consultar las métricas finales.*

10 documentos (3 impresos, 3 fotos y 4 manuscritos) y 10 clips (2 con TV de fondo, 3 de la conversación de trabajo y 5 de la familiar), tomados del corpus de la Parte 1. Las referencias son **transcripciones manuales**: las de los impresos y las fotos se copiaron del texto original (archivo de la tesis y PDF de SERVIR), y las de los manuscritos y los audios se tipearon a mano. En los manuscritos solo se transcribe lo escrito a mano (no el texto impreso de la agenda).

In [1]:
import json
import re
import time
import unicodedata
from pathlib import Path

import torch  # Se importa antes que el resto, como en la Parte 1
import numpy as np
import pandas as pd
import jiwer
import pytesseract
from PIL import Image, ImageOps
from faster_whisper import decode_audio
from collections import Counter

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SR = 16_000

c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Muestra de evaluación: 10 documentos de los 3 tipos y 10 clips de las 3 grabaciones
MUESTRA_DOCS = ["doc_002", "doc_005", "doc_007", "doc_009", "doc_010", "doc_011",
                "doc_017", "doc_018", "doc_023", "doc_025"]
MUESTRA_CLIPS = ["clip_001", "clip_005", "clip_007", "clip_010", "clip_013",
                 "clip_015", "clip_017", "clip_019", "clip_024", "clip_025"]

documentos = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
muestra_docs = documentos[documentos["id"].isin(MUESTRA_DOCS)].copy()
muestra_clips = clips[clips["id"].isin(MUESTRA_CLIPS)].copy()
print(muestra_docs["tipo"].value_counts().to_dict(), muestra_clips["fuente"].value_counts().to_dict())

{'manuscrito': 4, 'impreso': 3, 'foto': 3} {'Conversación familiar': 5, 'Conversación de trabajo': 3, 'Conversación con la hermana (TV de fondo)': 2}


In [3]:
# OCR alternativo: Tesseract 5.5 en español. Se le da la misma orientación que detectó el clasificador de
# PaddleOCR, porque su propio detector se equivocaba con las fotos giradas.
CARPETA_OCR = Path("datos/raw/documentos/ocr")
CARPETA_TESS = Path("datos/raw/documentos/tesseract")
CARPETA_TESS.mkdir(exist_ok=True)

def angulo_paddle(id_doc):
    """Ángulo (0, 90, 180 o 270) con que PaddleOCR enderezó el documento; 0 si no usó el clasificador."""
    with open(CARPETA_OCR / f"{id_doc}_res.json", encoding="utf-8") as f:
        return json.load(f).get("doc_preprocessor_res", {}).get("angle", 0)

textos, segundos = [], []
for fila in muestra_docs.itertuples():
    imagen = ImageOps.exif_transpose(Image.open(fila.archivo)).rotate(angulo_paddle(fila.id), expand=True)
    inicio = time.perf_counter()
    texto = pytesseract.image_to_string(imagen, lang="spa")
    segundos.append(round(time.perf_counter() - inicio, 1))
    textos.append(texto)
    (CARPETA_TESS / f"{fila.id}.txt").write_text(texto, encoding="utf-8")
    print(f"{fila.id}: {segundos[-1]} s")

muestra_docs["texto_tesseract"] = textos
muestra_docs["segundos_tesseract"] = segundos

doc_002: 2.0 s
doc_005: 4.0 s
doc_007: 2.8 s
doc_009: 7.1 s
doc_010: 13.1 s
doc_011: 9.6 s
doc_017: 6.6 s
doc_018: 4.8 s
doc_023: 6.5 s
doc_025: 8.4 s


In [4]:
# ASR alternativo: Qwen3-ASR 0.6B (Alibaba, Apache 2.0) en CPU, con el idioma fijado en español.
# Parakeet TDT 0.6B v3 se descartó: en una prueba previa mezclaba frases en inglés.
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration

CARPETA_QWEN = Path("datos/raw/audio/qwen3")
CARPETA_QWEN.mkdir(exist_ok=True)
proc_qwen = AutoProcessor.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf")
modelo_qwen = Qwen3ASRForConditionalGeneration.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf", dtype=torch.float32)

# Si un clip ya tiene su respaldo, no se vuelve a transcribir
for fila in muestra_clips.itertuples():
    respaldo = CARPETA_QWEN / f"{fila.id}.json"
    if respaldo.exists():
        print(f"{fila.id}: se lee el respaldo")
        continue
    audio = decode_audio(fila.archivo_audio, sampling_rate=SR)
    inicio = time.perf_counter()
    with torch.inference_mode():
        entrada = proc_qwen.apply_transcription_request(audio, language="es")
        salida = modelo_qwen.generate(**entrada, max_new_tokens=1024)
        texto = proc_qwen.decode(salida[0, entrada["input_ids"].shape[1]:], return_format="transcription_only").strip()
    segundos = time.perf_counter() - inicio
    respaldo.write_text(json.dumps({"id": fila.id, "texto": texto, "segundos_proceso": round(segundos, 1)},
                                   ensure_ascii=False), encoding="utf-8")
    print(f"{fila.id}: {fila.duracion_seg:.0f} s de audio en {segundos:.0f} s (RTF {segundos / fila.duracion_seg:.2f})")

Loading weights: 100%|██████████| 611/611 [00:21<00:00, 28.38it/s] 


clip_001: se lee el respaldo
clip_005: se lee el respaldo
clip_007: se lee el respaldo
clip_010: se lee el respaldo
clip_013: se lee el respaldo
clip_015: se lee el respaldo
clip_017: se lee el respaldo
clip_019: se lee el respaldo
clip_024: se lee el respaldo
clip_025: se lee el respaldo


## 2.2. Línea base sin ajustes

*Ejecutar la configuración base (PaddleOCR sin personalización adicional; faster-whisper small en CPU) sobre la muestra reservada y reportar CER (documentos) y WER (audio). Estaes la referencia contra la que compararán los modelos alternativos; no es obligatorio superarla.Normalicen las transcripciones de referencia y automáticas con el mismo criterio y, para WER,excluyan marcas como
[HABLANTE_1] que no sean palabras habladas.

**Configuraciones definidas antes de calcular las métricas:** PaddleOCR y faster-whisper `small`, tal como en la Parte 1 (se reutilizan sus resultados); Tesseract 5.5 (`lang="spa"`, con la misma orientación que detectó PaddleOCR); Qwen3-ASR 0.6B (idioma español). Para el CER y el WER, la referencia y la transcripción se normalizan igual: minúsculas, sin tildes, sin puntuación y sin marcas `[HABLANTE_n]`.

In [5]:
# Normalización común para referencias y transcripciones: sin marcas de hablante, minúsculas, sin tildes ni puntuación
def normalizar(texto):
    texto = re.sub(r"\[HABLANTE_\d+\]", " ", texto).lower()
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^\w\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

# Se comparan los textos crudos (sin seudonimizar), porque las referencias tienen los nombres tal como se oyen.
# Todo esto está en datos/raw/, que no se publica.
def leer_referencia(id_):
    return (Path("datos/raw/referencias") / f"{id_}.txt").read_text(encoding="utf-8")

def leer_json(ruta):
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

muestra_docs["referencia"] = muestra_docs["id"].apply(leer_referencia)
muestra_docs["texto_paddle"] = ["\n".join(leer_json(CARPETA_OCR / f"{i}_res.json")["rec_texts"]) for i in muestra_docs["id"]]

whisper = [leer_json(Path("datos/raw/audio/transcripciones") / f"{i}.json") for i in muestra_clips["id"]]
qwen = [leer_json(CARPETA_QWEN / f"{i}.json") for i in muestra_clips["id"]]
muestra_clips["referencia"] = muestra_clips["id"].apply(leer_referencia)
muestra_clips["texto_whisper"] = [" ".join(s["texto"] for s in w["segmentos"]) for w in whisper]
muestra_clips["texto_qwen"] = [q["texto"] for q in qwen]
muestra_clips["rtf_whisper"] = [w["segundos_proceso"] for w in whisper] / muestra_clips["duracion_seg"]
muestra_clips["rtf_qwen"] = [q["segundos_proceso"] for q in qwen] / muestra_clips["duracion_seg"]

In [6]:
def cer(referencia, hipotesis):
    return jiwer.cer(normalizar(referencia), normalizar(hipotesis) or "∅")

def wer(referencia, hipotesis):
    return jiwer.wer(normalizar(referencia), normalizar(hipotesis) or "∅")

def palabras_recuperadas(referencia, hipotesis):
    """Proporción de palabras de la referencia que aparecen en la transcripción, sin importar el orden."""
    ref, hip = Counter(normalizar(referencia).split()), Counter(normalizar(hipotesis).split())
    return sum((ref & hip).values()) / sum(ref.values())

for modelo in ["paddle", "tesseract"]:
    muestra_docs[f"cer_{modelo}"] = [cer(r, h) for r, h in zip(muestra_docs["referencia"], muestra_docs[f"texto_{modelo}"])]
    muestra_docs[f"recuperadas_{modelo}"] = [palabras_recuperadas(r, h) for r, h in zip(muestra_docs["referencia"], muestra_docs[f"texto_{modelo}"])]
for modelo in ["whisper", "qwen"]:
    muestra_clips[f"wer_{modelo}"] = [wer(r, h) for r, h in zip(muestra_clips["referencia"], muestra_clips[f"texto_{modelo}"])]

display(muestra_docs[["id", "tipo", "cer_paddle", "cer_tesseract", "recuperadas_paddle", "recuperadas_tesseract"]].round(3))
display(muestra_clips[["id", "fuente", "wer_whisper", "wer_qwen", "rtf_whisper", "rtf_qwen"]].round(2))

,id,tipo,cer_paddle,cer_tesseract,recuperadas_paddle,recuperadas_tesseract
1,doc_002,impreso,0.001,0.006,0.993,0.966
4,doc_005,impreso,0.000,0.002,1.000,0.994
6,doc_007,impreso,0.000,0.015,1.000,0.936
8,doc_009,foto,0.599,0.984,0.941,0.010
9,doc_010,foto,0.709,0.675,0.983,0.878
10,doc_011,foto,0.648,0.051,1.000,0.965
16,doc_017,manuscrito,0.177,0.913,0.456,0.018
17,doc_018,manuscrito,0.495,1.000,0.364,0.000
22,doc_023,manuscrito,0.118,0.754,0.667,0.039
24,doc_025,manuscrito,0.423,0.757,0.545,0.152


,id,fuente,wer_whisper,wer_qwen,rtf_whisper,rtf_qwen
0,clip_001,Conversación con la hermana (TV de fondo),0.23,0.25,1.01,2.05
4,clip_005,Conversación con la hermana (TV de fondo),0.42,0.44,1.15,1.59
6,clip_007,Conversación de trabajo,0.34,0.36,0.98,1.06
9,clip_010,Conversación de trabajo,0.57,0.52,0.85,6.86
12,clip_013,Conversación de trabajo,0.46,0.41,2.12,1.11
14,clip_015,Conversación familiar,0.32,0.27,1.06,1.86
16,clip_017,Conversación familiar,0.52,0.54,0.95,1.48
18,clip_019,Conversación familiar,0.47,0.48,1.04,2.14
23,clip_024,Conversación familiar,0.46,0.40,0.91,2.10
24,clip_025,Conversación familiar,0.42,0.34,0.84,2.36


In [7]:
# Métricas globales: jiwer suma los errores de los 10 casos y los divide entre el total de la referencia
def metrica_global(funcion, tabla, columna):
    return funcion(tabla["referencia"].apply(normalizar).tolist(),
                   tabla[columna].apply(normalizar).replace("", "∅").tolist())

CER_PADDLE = metrica_global(jiwer.cer, muestra_docs, "texto_paddle")
CER_TESSERACT = metrica_global(jiwer.cer, muestra_docs, "texto_tesseract")
WER_WHISPER = metrica_global(jiwer.wer, muestra_clips, "texto_whisper")
WER_QWEN = metrica_global(jiwer.wer, muestra_clips, "texto_qwen")
print(f"Documentos -> CER PaddleOCR (línea base): {CER_PADDLE:.3f} | CER Tesseract: {CER_TESSERACT:.3f}")
print(f"Audio -> WER faster-whisper small (línea base): {WER_WHISPER:.3f} (RTF {muestra_clips['rtf_whisper'].mean():.2f})"
      f" | WER Qwen3-ASR: {WER_QWEN:.3f} (RTF {muestra_clips['rtf_qwen'].mean():.2f})")

display(muestra_docs.groupby("tipo")[["cer_paddle", "cer_tesseract", "recuperadas_paddle", "recuperadas_tesseract"]].mean().round(3))
display(muestra_clips.groupby("fuente")[["wer_whisper", "wer_qwen", "rtf_whisper", "rtf_qwen"]].mean().round(2))

Documentos -> CER PaddleOCR (línea base): 0.402 | CER Tesseract: 0.500
Audio -> WER faster-whisper small (línea base): 0.423 (RTF 1.09) | WER Qwen3-ASR: 0.400 (RTF 2.26)


,cer_paddle,cer_tesseract,recuperadas_paddle,recuperadas_tesseract
tipo,,,,
foto,0.652,0.570,0.975,0.618
impreso,0.000,0.008,0.998,0.965
manuscrito,0.303,0.856,0.508,0.052


,wer_whisper,wer_qwen,rtf_whisper,rtf_qwen
fuente,,,,
Conversación con la hermana (TV de fondo),0.33,0.34,1.08,1.82
Conversación de trabajo,0.45,0.43,1.32,3.01
Conversación familiar,0.44,0.40,0.96,1.99


## 2.3 Extracción estructurada con LLM y auditoría humana

- *Usar un LLM (Ollama, Qwen local, Gemini API o similar) con un prompt estructurado para extraercampos JSON a partir de los textos OCR y ASR. Incluir al menos modalidad, tema y resumen; endocumentos, campos como emisor, fecha o monto solo si aparecen; en audios, citas_clave o numero_hablantes cuando se puedan sustentar. Usar null para datos ausentes; no inventar campos.*
- *Guardar datos/extraccion_llm.csv con: id, modalidad, campos_json, modelo_llm, fecha. Laconfianza autodeclarada por el LLM es opcional y no sustituye la auditoría.*
- *Auditoría humana: Revisar manualmente 20 extracciones al azar del corpus procesado(idealmente 10 documentos y 10 audios). Establecer criterios de corrección de campos y reportarla cantidad de extracciones completamente correctas sobre 20 (ej. 18/20 = 90 %) y ejemplos deerrores.*

Se usa Gemini (`gemini-3.8-flash`, API gratuita) con un prompt estructurado y un formato de respuesta fijo (`response_schema`). Por privacidad, se envían los textos **ya seudonimizados** de la Parte 1, así que ningún nombre real sale del computador. Campos: `modalidad`, `tema` y `resumen` en todos los casos; `emisor`, `fecha` y `monto` solo en documentos y solo si aparecen en el texto; `citas_clave` y `numero_hablantes` solo en audios. Los datos ausentes van como `null`, y el prompt prohíbe inventarlos. Como la cuota gratuita es de 20 pedidos al día, los 51 textos se envían en 6 lotes de hasta 9, y cada respuesta se guarda en `datos/raw/llm/` para no repetir pedidos.

El 25/10 el modelo respondió varias veces con el error 503 (saturado por alta demanda); el código reintenta automáticamente cada 2 minutos hasta completar cada lote. Otros modelos de la lista (`gemini-3.7-flash`, `gemini-3.6-flash`) tampoco estaban disponibles o no tenían cuota gratuita.

In [8]:
import os
from datetime import date
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel
from google import genai
from google.genai import types

load_dotenv()  # lee la clave del archivo .env (que no se publica)
MODELO_LLM = "gemini-3.5-flash"  # gemini-3.8-flash: error 503 (saturado) y luego cuota diaria agotada el 25/10
cliente = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("¿Clave de Gemini cargada?", os.getenv("GEMINI_API_KEY") is not None)

# Formato de respuesta obligatorio: Gemini debe devolver exactamente estos campos por cada texto
class Extraccion(BaseModel):
    id: str
    modalidad: Literal["documento", "audio"]
    tema: str
    resumen: str
    emisor: str | None
    fecha: str | None
    monto: str | None
    citas_clave: list[str] | None
    numero_hablantes: int | None

INSTRUCCIONES = """Eres un analista que convierte textos peruanos en datos estructurados.
Recibirás varios textos, cada uno con su id y su modalidad: "documento" (texto leído con OCR de una imagen) o
"audio" (transcripción automática de una conversación; las marcas [HABLANTE_n] indican quién habla, y
[PERSONA_n] o [APELLIDO] reemplazan nombres reales).
Para cada texto devuelve:
- id y modalidad: copiados tal cual.
- tema: el tema principal, en pocas palabras.
- resumen: una o dos oraciones con lo esencial.
- emisor, fecha y monto: SOLO en documentos y SOLO si aparecen escritos en el texto (cópialos tal como aparecen;
  en monto, el más importante). Si no aparecen, null. En audios, siempre null.
- citas_clave: SOLO en audios, de 1 a 3 frases textuales e importantes de la transcripción. En documentos, null.
- numero_hablantes: SOLO en audios, el número de marcas [HABLANTE_n] distintas. En documentos, null.
Reglas: no inventes datos ni completes con suposiciones; si un dato no está en el texto, usa null.
El texto puede tener errores de OCR o de transcripción: en las citas, cópialo tal cual."""

¿Clave de Gemini cargada? True


In [9]:
PEDIR_A_GEMINI = False  # Gemini se reemplazó por Claude (ver nota de 2.3); sus lotes parciales quedan en datos/raw/llm/
# Se juntan los 51 textos seudonimizados de la Parte 1 (documentos y clips completos, no solo la muestra)
textos = pd.concat([
    pd.DataFrame({"id": documentos["id"], "modalidad": "documento", "texto": documentos["texto_ocr"]}),
    pd.DataFrame({"id": clips["id"], "modalidad": "audio", "texto": clips["texto_asr"]}),
], ignore_index=True)

CARPETA_LLM = Path("datos/raw/llm") / MODELO_LLM
CARPETA_LLM.mkdir(exist_ok=True)
TAMANO_LOTE = 9  # 51 textos -> 6 pedidos

from google.genai import errors

def extraer_lote(lote, intentos=15):
    """Envía un grupo de textos a Gemini y devuelve sus extracciones como diccionarios.
    Si Gemini está saturado (error 503), espera y reintenta: 30 s, 60 s, 90 s..."""
    bloques = "\n\n".join(f"### id: {f.id} | modalidad: {f.modalidad}\n{f.texto}" for f in lote.itertuples())
    for intento in range(1, intentos + 1):
        try:
            respuesta = cliente.models.generate_content(
                model=MODELO_LLM,
                contents=INSTRUCCIONES + "\n\nTextos:\n\n" + bloques,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=list[Extraccion],
                ),
            )
            return [e.model_dump() for e in respuesta.parsed]
        except errors.ServerError as error:
            espera = 120 * intento
            print(f"   Gemini saturado ({error.code}), intento {intento}: se espera {espera} s")
            time.sleep(espera)
    raise RuntimeError("Gemini sigue saturado después de 6 intentos: probar más tarde u otro modelo")

# Si un lote ya tiene su respaldo, no se vuelve a pedir (así no se gasta la cuota)
for k, inicio in enumerate(range(0, len(textos), TAMANO_LOTE), 1):
    respaldo = CARPETA_LLM / f"lote_{k:02d}.json"
    if respaldo.exists() or not PEDIR_A_GEMINI:
        print(f"Lote {k}: se lee el respaldo")
        continue
    lote = textos.iloc[inicio:inicio + TAMANO_LOTE]
    resultado = extraer_lote(lote)
    respaldo.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Lote {k}: {len(resultado)} extracciones de {len(lote)} textos")
    time.sleep(10)  # pausa entre pedidos

Lote 1: se lee el respaldo
Lote 2: se lee el respaldo
Lote 3: se lee el respaldo
Lote 4: se lee el respaldo
Lote 5: se lee el respaldo
Lote 6: se lee el respaldo


Debido a que se presentó dificultad con gemini-3.8-flash,gemini-3.7-flash y gemini-3.6-flash, se optó por continuar con Claude.

In [10]:
# Extracción con Claude (API de Anthropic), porque el 25/10 Gemini estaba saturado (503) y agotó su cuota gratuita (429)
import anthropic

from dotenv import load_dotenv

# Los 51 textos seudonimizados de la Parte 1 (igual que en la celda de Gemini)
textos = pd.concat([
    pd.DataFrame({"id": documentos["id"], "modalidad": "documento", "texto": documentos["texto_ocr"]}),
    pd.DataFrame({"id": clips["id"], "modalidad": "audio", "texto": clips["texto_asr"]}),
], ignore_index=True)
TAMANO_LOTE = 9

load_dotenv(override=True)  # vuelve a leer el .env, que ahora tiene ANTHROPIC_API_KEY
MODELO_LLM = "claude-opus-5-5"
cliente_claude = anthropic.Anthropic()  # toma la clave de la variable ANTHROPIC_API_KEY

class Lote(BaseModel):
    extracciones: list[Extraccion]

def extraer_lote(lote):
    """Envía un grupo de textos a Claude y devuelve sus extracciones como diccionarios."""
    bloques = "\n\n".join(f"### id: {f.id} | modalidad: {f.modalidad}\n{f.texto}" for f in lote.itertuples())
    respuesta = cliente_claude.messages.parse(
        model=MODELO_LLM,
        max_tokens=16000,
        system=INSTRUCCIONES,
        messages=[{"role": "user", "content": "Textos:\n\n" + bloques}],
        output_format=Lote,  # formato de respuesta obligatorio (salida estructurada)
    )
    return [e.model_dump() for e in respuesta.parsed_output.extracciones]

# Una carpeta de respaldos por modelo; si un lote ya está guardado, no se vuelve a pedir
CARPETA_LLM = Path("datos/raw/llm") / MODELO_LLM
CARPETA_LLM.mkdir(parents=True, exist_ok=True)
for k, inicio in enumerate(range(0, len(textos), TAMANO_LOTE), 1):
    respaldo = CARPETA_LLM / f"lote_{k:02d}.json"
    if respaldo.exists():
        print(f"Lote {k}: se lee el respaldo")
        continue
    lote = textos.iloc[inicio:inicio + TAMANO_LOTE]
    resultado = extraer_lote(lote)
    respaldo.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Lote {k}: {len(resultado)} extracciones de {len(lote)} textos")

Lote 1: se lee el respaldo
Lote 2: se lee el respaldo
Lote 3: se lee el respaldo
Lote 4: se lee el respaldo
Lote 5: se lee el respaldo
Lote 6: se lee el respaldo


In [11]:
# Se reúnen los lotes en la tabla pedida: id, modalidad, campos_json, modelo_llm y fecha
filas = []
for respaldo in sorted(CARPETA_LLM.glob("lote_*.json")):
    fecha_lote = date.fromtimestamp(respaldo.stat().st_mtime).isoformat()  # día en que se hizo el pedido
    for e in json.loads(respaldo.read_text(encoding="utf-8")):
        filas.append({"id": e["id"], "modalidad": e["modalidad"],
                      "campos_json": json.dumps(e, ensure_ascii=False),
                      "modelo_llm": MODELO_LLM, "fecha": fecha_lote})

extraccion = pd.DataFrame(filas)
print("Extracciones:", len(extraccion), "| textos sin extracción:", sorted(set(textos["id"]) - set(extraccion["id"])))
extraccion.to_csv("datos/extraccion_llm.csv", index=False, encoding="utf-8")
extraccion.head()

Extracciones: 51 | textos sin extracción: []


,id,modalidad,campos_json,modelo_llm,fecha
0,doc_001,documento,"{""id"": ""doc_001"", ""modalidad"": ""documento"", ""t...",claude-opus-5-5,2026-10-25
1,doc_002,documento,"{""id"": ""doc_002"", ""modalidad"": ""documento"", ""t...",claude-opus-5-5,2026-10-25
2,doc_003,documento,"{""id"": ""doc_003"", ""modalidad"": ""documento"", ""t...",claude-opus-5-5,2026-10-25
3,doc_004,documento,"{""id"": ""doc_004"", ""modalidad"": ""documento"", ""t...",claude-opus-5-5,2026-10-25
4,doc_005,documento,"{""id"": ""doc_005"", ""modalidad"": ""documento"", ""t...",claude-opus-5-5,2026-10-25


### Auditoría humana (20 extracciones al azar: 10 documentos y 10 audios)

**Criterio:** una extracción es **completamente correcta** solo si cumple todo lo siguiente: (1) `tema` y `resumen` describen fielmente el texto, sin agregar información que no está en él; (2) `emisor`, `fecha` y `monto` tienen valor solo si ese dato aparece en el texto y es el que corresponde, y son `null` si no aparece; (3) en audios, las `citas_clave` aparecen textualmente y son relevantes, y `numero_hablantes` coincide con las marcas `[HABLANTE_n]`. Un solo error en cualquier campo hace que toda la extracción cuente como incorrecta.

In [12]:
# Auditoría: 10 documentos y 10 audios elegidos al azar (semilla fija, para que se pueda repetir)
extraccion = pd.read_csv("datos/extraccion_llm.csv")
auditoria = pd.concat([
    extraccion[extraccion["modalidad"] == "documento"].sample(10, random_state=42),
    extraccion[extraccion["modalidad"] == "audio"].sample(10, random_state=42),
])
texto_por_id = dict(zip(textos["id"], textos["texto"]))

for fila in auditoria.itertuples():
    campos = json.loads(fila.campos_json)
    print("=" * 100)
    print(f"{fila.id} ({fila.modalidad})\nTEXTO: {texto_por_id[fila.id]}\nEXTRACCIÓN:")
    for campo in ["tema", "resumen", "emisor", "fecha", "monto", "citas_clave", "numero_hablantes"]:
        print(f"   {campo}: {campos[campo]}")

doc_009 (documento)
TEXTO: Ingresos de las personas al servicio de los Gobiernos Locales
son las que están contratadas por proyectos de inversión
(ingreso mensual promedio de S/ 1,939) seguidas por las
contratadas bajo el D. Leg. 276 (S/ 1,923). Luego, con una
brecha de aproximadamente S/ 600, están las personas
contratadas por Locación de servicios (S/ 1,338). Los
ingresos promedio más bajos corresponden a las
Este capitulo brinda información referente a los ingresos²
personas contratadas bajo CAS y D. Leg. 728 (S/ 1,293 y
de las personas al servicio de los gobiernos locales. Se
S/1,263, respectivamente).
muestra información como el ingreso mensual bruto
minimo, máximo y promedio, desagregado por tipo de
El ingreso mediano registra valores cercanos y menores
modalidad de contratación, por la naturaleza de los
al promedio para todos los regimenes con excepción de
órganos y por el sexo de los ocupantes.
las personas contratadas por Proyectos de inversión, con
una diferencia que va de S/

In [13]:
# Resultado de la revisión manual: (1 = completamente correcta, 0 = al menos un error, comentario)
REVISION = {
    "doc_009": (1, "se transcribió bien, aunque desordenado. Pero sí aplicó un bien resumen"),
    "doc_017": (1, "hubo errores en OCR, pero sí aplicó bien el resumen"),
    "doc_001": (1, "Excelente OCR y resumen"),
    "doc_024": (1, "hubo errores en OCR, pero sí aplicó bien el resumen"),
    "doc_012": (1, "se transcribió bien, aunque desordenado. Pero sí aplicó un bien resumen"),
    "doc_010": (1, "se transcribió bien, aunque desordenado. Pero sí aplicó un bien resumen"),
    "doc_014": (1, "se transcribió bien, aunque desordenado. Pero sí aplicó un bien resumen"),
    "doc_002": (1, "Excelente OCR y resumen"),
    "doc_023": (1, "hubo errores en OCR, pero sí aplicó bien el resumen"),
    "doc_006": (1, "Excelente OCR y resumen"),
    "clip_009": (0, "El resumen habla del campo 'responsable asignado': interpretó mal 'a Zaire' (en el audio, el aplicativo ASAI), un error que viene del ASR"),
    "clip_017": (1, "Errores previos de la diarización (hablantes repartidos en 4 marcas) y del ASR ('la vela' por Falabella); pero el resumen fue preciso"),
    "clip_001": (1, "Buena transcripción, buen resumen identificó bien los hablantes"),
    "clip_025": (1, "excelente resumen e identificó bien a los hablantes, pese a que una intervino muy poco, la consideró"),
    "clip_012": (1, "excelente resumen e identificó bien a los hablantes"),
    "clip_010": (1, "excelente resumen e identificó bien a los hablantes"),
    "clip_014": (1, "se inventó textos, pero referente al resumen, citas clave y número de hablates, está ok"),
    "clip_002": (1, "buena transcripción, citas y hablantes, buen resumen"),
    "clip_024": (1, "excelente resumen e identificó bien el numero de hablantes y citas claves"),
    "clip_006": (1, "excelente resumen e identificó bien el numero de hablantes y citas claves"),
}
auditoria["correcta"] = auditoria["id"].map(lambda i: REVISION[i][0])
auditoria["comentario"] = auditoria["id"].map(lambda i: REVISION[i][1])
ACUERDO = auditoria["correcta"].sum()
print(f"Acuerdo LLM-Humano: {ACUERDO}/20 = {ACUERDO / 20:.0%}")
display(auditoria.loc[auditoria["correcta"] == 0, ["id", "modalidad", "comentario"]])
auditoria[["id", "modalidad", "correcta", "comentario"]].to_csv("datos/auditoria_llm.csv", index=False, encoding="utf-8")

Acuerdo LLM-Humano: 19/20 = 95%


,id,modalidad,comentario
33,clip_009,audio,El resumen habla del campo 'responsable asigna...


## 2.4. Mejora y comparación de modelos
- *Documentos: comparar PaddleOCR con un OCR alternativo , por ejemplo Tesseract, u otromodelo de lectura de texto debidamente justificado, midiendo CER en los mismos 10 documentos.*
- *Audio: comparar faster-whisper con otro ASR (Qwen3-ASR, Parakeet o MMS si corresponde alidioma), midiendo WER y latencia en los mismos 10 clips. El factor de tiempo real se obtienedividiendo el tiempo de procesamiento entre la duración del audio.*

In [14]:
# Tabla comparativa sobre las mismas muestras (10 documentos y 10 clips). La latencia de los ASR es la mediana del RTF
# (tiempo de proceso / duración del audio), para que no pese el valor atípico de Qwen en clip_010.
comparativa = pd.DataFrame([
    {"modalidad": "Documentos", "modelo": "PaddleOCR 3.7 (PP-OCRv6 medium) – línea base", "metrica": "CER",
     "valor": CER_PADDLE, "palabras_recuperadas": muestra_docs["recuperadas_paddle"].mean(),
     "latencia": "60–216 s por imagen (CPU, Parte 1)", "costo": "Gratis, local"},
    {"modalidad": "Documentos", "modelo": "Tesseract 5.5 (spa)", "metrica": "CER",
     "valor": CER_TESSERACT, "palabras_recuperadas": muestra_docs["recuperadas_tesseract"].mean(),
     "latencia": f"{muestra_docs['segundos_tesseract'].mean():.0f} s por imagen (CPU)", "costo": "Gratis, local"},
    {"modalidad": "Audio", "modelo": "faster-whisper small (int8, VAD) – línea base", "metrica": "WER",
     "valor": WER_WHISPER, "palabras_recuperadas": None,
     "latencia": f"RTF {muestra_clips['rtf_whisper'].median():.2f}", "costo": "Gratis, local"},
    {"modalidad": "Audio", "modelo": "Qwen3-ASR 0.6B", "metrica": "WER",
     "valor": WER_QWEN, "palabras_recuperadas": None,
     "latencia": f"RTF {muestra_clips['rtf_qwen'].median():.2f}", "costo": "Gratis, local"},
    {"modalidad": "Extracción LLM", "modelo": "Claude Opus 5.5", "metrica": "Acuerdo con la auditoría",
     "valor": ACUERDO / 20, "palabras_recuperadas": None,
     "latencia": "≈15–20 s por lote de 9 textos", "costo": "Menos de US$ 1 (API de pago)"},
])
comparativa[["valor", "palabras_recuperadas"]] = comparativa[["valor", "palabras_recuperadas"]].round(3)
comparativa

,modalidad,modelo,metrica,valor,palabras_recuperadas,latencia,costo
0,Documentos,PaddleOCR 3.7 (PP-OCRv6 medium) – línea base,CER,0.402,0.795,"60–216 s por imagen (CPU, Parte 1)","Gratis, local"
1,Documentos,Tesseract 5.5 (spa),CER,0.500,0.496,6 s por imagen (CPU),"Gratis, local"
2,Audio,"faster-whisper small (int8, VAD) – línea base",WER,0.423,NaN,RTF 1.00,"Gratis, local"
3,Audio,Qwen3-ASR 0.6B,WER,0.400,NaN,RTF 1.96,"Gratis, local"
4,Extracción LLM,Claude Opus 5.5,Acuerdo con la auditoría,0.950,NaN,≈15–20 s por lote de 9 textos,Menos de US$ 1 (API de pago)


OCR: PaddleOCR (0,40) es mejor que Tesseract (0,50), y la diferencia real es mayor de lo que muestra el CER. PaddleOCR recupera el 80 % de las palabras y Tesseract solo el 50 %. El CER de PaddleOCR sube porque desordena las páginas de dos columnas, no porque lea mal. Tesseract, en cambio, es mucho más rápido (6 s frente a 1–3,5 min por imagen). En tu tabla por tipo del paso 2.2 se ve el detalle:
Impresos: los dos son casi perfectos (CER 0,000 y 0,008).
Fotos: Tesseract tiene menor CER (0,57 frente a 0,65), pero PaddleOCR encuentra el 97 % de las palabras y Tesseract solo el 62 %.
Manuscritos: PaddleOCR gana con claridad (0,30 frente a 0,86).
ASR: prácticamente empate. Qwen (0,40) tiene 2 puntos menos de WER que Whisper (0,42). Con solo 10 clips, esa diferencia no es concluyente. Whisper es el doble de rápido (RTF 1,00 frente a 1,96), y además da tiempos y puntuación, que se necesitan para los SRT y la diarización.
LLM: Claude acertó en 19 de 20 extracciones, y su único error vino de una palabra mal transcrita por el ASR.

In [16]:
# Se completa texto_referencia en los CSV de la Parte 1 (seudonimizado; aquí no se distingue mayúsculas,
# porque las referencias se tipearon en minúsculas) y se guardan las métricas por caso para el dashboard
SEUDONIMOS = pd.read_csv("datos/raw/seudonimos.csv", encoding="utf-8")

def seudonimizar(texto):
    for nombre, seudonimo in zip(SEUDONIMOS["nombre"], SEUDONIMOS["seudonimo"]):
        texto = re.sub(rf"\b{re.escape(nombre)}\b", seudonimo, texto, flags=re.IGNORECASE)
    return texto

documentos["texto_referencia"] = documentos["id"].map(dict(zip(muestra_docs["id"], muestra_docs["referencia"].apply(seudonimizar))))
clips["texto_referencia"] = clips["id"].map(dict(zip(muestra_clips["id"], muestra_clips["referencia"].apply(seudonimizar))))
documentos.to_csv("datos/documentos.csv", index=False, encoding="utf-8")
clips.to_csv("datos/transcripciones.csv", index=False, encoding="utf-8")

muestra_docs[["id", "tipo", "cer_paddle", "cer_tesseract", "recuperadas_paddle", "recuperadas_tesseract",
              "segundos_tesseract"]].to_csv("datos/evaluacion_ocr.csv", index=False, encoding="utf-8")
muestra_clips[["id", "fuente", "duracion_seg", "wer_whisper", "wer_qwen", "rtf_whisper", "rtf_qwen"]].to_csv(
    "datos/evaluacion_asr.csv", index=False, encoding="utf-8")
comparativa.to_csv("datos/comparativa_modelos.csv", index=False, encoding="utf-8")
print("Referencias completadas:", documentos["texto_referencia"].notna().sum(), "documentos y",
      clips["texto_referencia"].notna().sum(), "clips")

Referencias completadas: 10 documentos y 10 clips


## 2.5. Casos difíciles

*Analizar el desempeño en manuscritos, audio con ruido o música y (opcional,bonificación) quechua de Huaylas con MMS.*

- **Manuscritos:** es el tipo de documento más difícil. PaddleOCR obtiene un CER de 0,30 y recupera el 51 % de las palabras; Tesseract, un CER de 0,86 y apenas el 5 %. El peor caso es `doc_018` (CER 0,50 con PaddleOCR), una página con notas al margen y flechas, sin un orden de lectura lineal. Las páginas con fórmulas (`doc_019`–`doc_021`) se dejaron fuera de la muestra porque no se pueden transcribir con texto plano. Aun así, el LLM compensa parte de los errores: en `doc_017` el OCR leyó "feorema do Bayes" y el resumen dice correctamente "teorema de Bayes".
- **Fotos con inclinación y dos columnas:** PaddleOCR lee bien los caracteres (recupera el 97,5 % de las palabras), pero intercala las líneas de las dos columnas, y eso eleva su CER a 0,65. Tesseract respeta las columnas cuando la página está derecha (`doc_011`: CER 0,05), pero no lee la foto inclinada `doc_009` (CER 0,98). Además, el clasificador de orientación de PaddleOCR puso `doc_009` de cabeza en la Parte 1.
- **Audio con ruido de fondo (TV encendida, `clip_001` y `clip_005`):** contra lo esperado, no es el peor grupo (WER 0,33 con Whisper y 0,34 con Qwen). Más difíciles fueron la conversación de trabajo (0,45 y 0,43), con jerga y nombres de sistemas ("ASAI", "D100", "expedientes"), y la familiar (0,44 y 0,40), con varias personas hablando a la vez. El único error de la auditoría viene de ahí: Whisper escribió "a Zaire" en vez de "ASAI", y el LLM lo interpretó como "responsable asignado".
- **Diarización:** con varias voces superpuestas, k-means reparte a una misma persona en dos marcas (por ejemplo, en `clip_017`). Además, como se le indica que son 4 personas, siempre forma 4 grupos, aunque en un clip hablen menos.
- **Otro ASR descartado:** Parakeet TDT 0.6B v3, el más rápido en la prueba previa (RTF 0,3–0,4), mezclaba frases en inglés en el español peruano coloquial, porque no permite fijar el idioma.



## 2.6. Comparativa y decisión

*Tabla comparativa de todos los modelos (base vs. mejorado) y justificaciónde cuál elegirían para un entorno real considerando precisión, costo computacional y latencia. Incluirreflexión por tipo de documento y por tipo de audio: identificar dónde hubo mejor/peor desempeñoy explicar la causa.*

**Documentos:** para un entorno real con CPU, se elegiría un esquema mixto: Tesseract para los impresos limpios (CER 0,008, en unos 6 s por imagen) y PaddleOCR para las fotos y los manuscritos, donde recupera muchas más palabras (97 % frente a 62 % en fotos y 51 % frente a 5 % en manuscritos), aunque tarde de 1 a 3,5 minutos por imagen. Para bajar su CER habría que agregar una etapa que reconstruya el orden de lectura de las páginas a dos columnas. Con una GPU, PaddleOCR podría usarse para todo.

**Audio:** se elige faster-whisper `small`. Su WER (0,42) es casi igual al de Qwen3-ASR (0,40): 2 puntos en 10 clips no es una diferencia concluyente. En cambio, Whisper es el doble de rápido (RTF 1,00 frente a 1,96) y entrega tiempos por segmento y puntuación, que se necesitan para los subtítulos y la diarización. Qwen3-ASR sería la opción si la precisión fuera la prioridad y se contara con una GPU.

**Extracción:** Claude Opus 5.5 logró un acuerdo de 19/20 con la auditoría, por menos de US$ 1 en todo el corpus. La API gratuita de Gemini no fue confiable ese día (saturación y cuota agotada), así que en un entorno real convendría una API de pago o un modelo local.

**Reflexión por tipo:** el mejor desempeño está en los impresos (CER ≈ 0 con ambos OCR) y en el audio tranquilo de dos personas (`clip_001`: WER 0,23). El peor, en los manuscritos con diagramas (`doc_018`) y en la conversación de trabajo con jerga y voces superpuestas (`clip_010`: WER 0,57 con Whisper). Las causas son la letra y el orden no lineal, en los documentos, y el vocabulario técnico y el habla simultánea, en los audios.

**Resultados para la entrega (modelos seleccionados):** CER 0,40 (PaddleOCR) | WER 0,42 (faster-whisper small) | Acuerdo LLM-Humano 19/20.
